# Clio-Infra — downloads

Pulls every indicator workbook and description page from `clio-infra.eu`, then
measures what arrived.

**Input** — `https://clio-infra.eu/`: the index page, then
`data/<file>_Compact.xlsx` and `Indicators/<page>.html` for each of 86
indicators, plus `data/DataAtHistoricalBorders.xlsx`. No auth, no bot filter.
Licence **CC0-1.0** (checked on DataverseNL for three indicators).

**Output** — everything under `notebooks/clio/data/`, which is **gitignored**:

| path | what |
| --- | --- |
| `index.html` | the index page, as fetched |
| `raw/*.xlsx` | 87 workbooks, ~36 MB, names exactly as published |
| `pages/*.html` | 86 indicator description pages |
| `downloads.json` | manifest: category, file, `Last-Modified`, sha256, published span and count, the page's labelled panels |
| `inventory.json` | what the files actually hold — see `inventory.ipynb` |
| `timeseries/clio.jsonl` | 11,042 stored series, one per indicator per country — what `load_timeseries` reads |
| `clio_series_<category>.yaml` | the catalog, one file per Clio category — what `load_catalog` reads |
| `units.yaml` | a unit per indicator — the workbooks state none, so each comes from the DataverseNL deposit filed under `notebooks/clio_historical`: its workbook's row 2, else its working paper. Never inferred. |

**Re-running.** The fetch is idempotent: files already on disk are skipped, so
nothing needs deleting first. Pass `refresh=True` to re-pull everything — worth
doing only if `Last-Modified` moves. Every file carried the same stamp,
`2024-09-02 13:46:35 GMT`, when first pulled: one publication, refreshed as a
unit.

A full pull is 173 requests at one second apart — about six minutes.

In [1]:
import sys
sys.path.append("../")
sys.path.append("../../")

from utils import catalog, fetch, inventory, units

In [2]:
counts = fetch.fetch_all()          # refresh=True to re-pull
counts

{'indicators': 86, 'downloaded': 0, 'cached': 87, 'pages': 0}

In [3]:
measured = inventory.build()
print(f"{len(measured)} indicators measured -> {fetch.DATA_DIR / inventory.INVENTORY}")

86 indicators measured -> /Users/troy/Develop/gly.fish/meida/notebooks/clio/data/inventory.json


## Units

From `notebooks/clio_historical/data/documentation.yaml`, which that source's
`documentation.build()` writes from the deposits filed there. Workbook row 2 is
preferred, the working paper's unit is the fallback. Re-run after filing more.


In [4]:
counts = units.harvest()
print(f"{counts['with_unit']} of {counts['indicators']} filed indicators have a unit "
      f"({counts['from_workbook']} from row 2); {len(measured) - counts['indicators']} indicators not filed")


69 of 76 filed indicators have a unit (65 from row 2); 10 indicators not filed


## Build the series and catalog

One series per indicator per country. A row with a blank country code takes the
code its name carries elsewhere; where that lands on a year the coded series
already has, the coded row wins and the clash is listed here — Canada's two
1946-47 real-wage values are the United States' numbers.

In [5]:
report = catalog.export()
print(f"{report['series']:,} series, {report['observations']:,} observations, "
      f"{report['catalog_files']} catalog files; {report['with_units']:,} series have a unit")
for slug, country, year, dropped, kept in report["collisions"]:
    print(f"  clash: {slug} {country} {year}: kept {kept}, dropped {dropped}")

11,042 series, 902,287 observations, 11 catalog files; 8,820 series have a unit
  clash: LabourersRealWage Canada 1946: kept 40.99347307, dropped 66.73118859
  clash: LabourersRealWage Canada 1947: kept 41.15988672, dropped 57.35376473
  clash: LabourersRealWage Morocco 1956: kept 8.64311871, dropped 8.64311871
  clash: LabourersRealWage Morocco 1957: kept 9.57991558, dropped 9.57991558


## Load into Postgres

`time_series_source` holds the observations, `series_catalog` the discovery
rows. Both loads upsert — re-running changes nothing unless the build did — and
both write in batches, since 11,042 series exceed what one statement can bind.
The catalog load prunes only `source='clio'` rows.

In [6]:
from db_import import load_catalog, load_timeseries

ts = load_timeseries.load_all(fetch.DATA_DIR / "timeseries")["clio.jsonl"]
print(f"time_series_source: {ts['series']:,} series — {len(ts['new']):,} new, "
      f"{len(ts['changed']):,} changed, {ts['observations_moved']:,} observations moved, "
      f"{len(ts['relabelled']):,} relabelled")
print("series_catalog    :", load_catalog.load(fetch.DATA_DIR, "clio"))

time_series_source: 11,042 series — 0 new, 0 changed, 0 observations moved, 0 relabelled


series_catalog    : {'written': 11042, 'pruned': 0}
